In [11]:
import os
import numpy as np
import matplotlib.pyplot as plt
from skimage.measure import label, regionprops, find_contours
from scipy.ndimage import gaussian_filter
import rasterio
from rasterio.mask import mask
from rasterio.warp import transform_bounds
from shapely.geometry import box
import geopandas as gpd
from matplotlib.patches import Circle
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib import colormaps

In [12]:
os.environ['AWS_NO_SIGN_REQUEST'] = 'YES'

In [13]:
def load_band(band_path, bbox):
    try:
        with rasterio.open(band_path) as src:
            bbox_transformed = transform_bounds('EPSG:4326', src.crs, *bbox)
            bbox_geom = [{
                "type": "Polygon",
                "coordinates": [[
                    [bbox_transformed[0], bbox_transformed[1]],
                    [bbox_transformed[0], bbox_transformed[3]],
                    [bbox_transformed[2], bbox_transformed[3]],
                    [bbox_transformed[2], bbox_transformed[1]],
                    [bbox_transformed[0], bbox_transformed[1]]
                ]]
            }]
            out_image, out_transform = mask(src, bbox_geom, crop=True)
            if out_image.size == 0:
                raise ValueError("Clipped band is empty. Ensure the bounding box overlaps the data.")
            return out_image[0], out_transform, src.crs
    except Exception as e:
        print(f"Error loading or clipping file: {band_path} - {e}")
        return None, None, None

In [14]:
def calculate_ndvi(nir, red):
    np.seterr(divide='ignore', invalid='ignore')
    ndvi = (nir - red) / (nir + red)
    ndvi[(nir + red) == 0] = np.nan  # Handle division by zero
    return ndvi

In [15]:
def calculate_total_acreage(labeled_fields, transform):
    pixel_size = abs(transform[0])  
    pixel_area_m2 = pixel_size ** 2
    total_area_m2 = (labeled_fields > 0).sum() * pixel_area_m2
    return total_area_m2 / 4046.86  # Convert square meters to acres

In [16]:
def visualize_ndvi_and_fields(ndvi, labeled_fields, field_properties, num_fields_to_visualize=5):
    """Visualize NDVI and individual fields with acreage annotations."""
    try:
        if np.nanmin(ndvi) == np.nanmax(ndvi):
            print("NDVI has no range; skipping visualization.")
            return

        # Full NDVI map with field contours
        plt.figure(figsize=(12, 8))
        ndvi_plot = plt.imshow(ndvi, cmap="RdYlGn", vmin=-1, vmax=1)
        plt.colorbar(ndvi_plot, label="NDVI")
        for field in field_properties:
            field_mask = labeled_fields == field.label
            plt.contour(field_mask, colors="yellow", linewidths=0.5)
        plt.title("NDVI Map with Field Contours")
        plt.show()

        # Individual field visualizations in chunks
        for i, field in enumerate(field_properties[:num_fields_to_visualize]):
            minr, minc, maxr, maxc = field.bbox
            isolated_ndvi = np.where(labeled_fields == field.label, ndvi, np.nan)

            # Field metrics
            field_area_m2 = field.area * (10 ** 2)  # Assuming 10m pixel resolution
            field_area_acres = field_area_m2 / 4046.86
            shape_complexity = (field.perimeter ** 2) / field.area

            plt.figure(figsize=(8, 6))
            plt.imshow(isolated_ndvi[minr:maxr, minc:maxc], cmap="RdYlGn", vmin=-1, vmax=1)
            plt.title(f"Field {i+1}: Area = {field_area_acres:.2f} ac, Shape = {shape_complexity:.2f}")
            plt.colorbar(label="NDVI")
            plt.show()
    except Exception as e:
        print(f"Error during visualization: {e}")

In [17]:
def process_ndvi_fields(red_band_path, nir_band_path, bbox, ndvi_threshold=0.4, min_area_pixels=100, num_fields_to_visualize=5):
    """Process Sentinel-2 data and calculate NDVI fields with total acreage."""
    red_band, red_transform, _ = load_band(red_band_path, bbox)
    nir_band, nir_transform, _ = load_band(nir_band_path, bbox)
    if red_band is None or nir_band is None:
        print("Error: Failed to load bands.")
        return

    
    ndvi = calculate_ndvi(nir_band, red_band)
    print(f"NDVI calculated. NDVI range: {np.nanmin(ndvi)} to {np.nanmax(ndvi)}")

    
    crop_mask = ndvi > ndvi_threshold
    labeled_fields, num_fields = label(crop_mask, connectivity=2, return_num=True)
    field_properties = [
        prop for prop in regionprops(labeled_fields)
        if prop.area >= min_area_pixels
    ]
    print(f"Number of fields detected: {num_fields}")
    print(f"Number of fields after filtering: {len(field_properties)}")

    
    total_acreage = calculate_total_acreage(labeled_fields, red_transform)
    print(f"Total acreage for Huron County: {total_acreage:.2f} acres")

    
    visualize_ndvi_and_fields(ndvi, labeled_fields, field_properties, num_fields_to_visualize=num_fields_to_visualize)

In [ ]:
# Define bounding box for Huron County
huron_county_bbox = (-83.5941, 43.6764, -82.5872, 44.0789)


red_band_path = "/vsis3/sentinel-cogs/sentinel-s2-l2a-cogs/17/T/LJ/2024/9/S2A_17TLJ_20240915_0_L2A/B04.tif"
nir_band_path = "/vsis3/sentinel-cogs/sentinel-s2-l2a-cogs/17/T/LJ/2024/9/S2A_17TLJ_20240915_0_L2A/B08.tif"

# Run the process
process_ndvi_fields(
    red_band_path, nir_band_path, huron_county_bbox, ndvi_threshold=0.4, min_area_pixels=100, num_fields_to_visualize=5
)

NDVI calculated. NDVI range: 0.0 to 394.6024096385542
Number of fields detected: 19268
Number of fields after filtering: 423
Total acreage for Huron County: 673243.75 acres
